In [32]:
# pip install snowflake

#**Setup & Config**

In [33]:
# --- CELL 1: SETUP, IMPORTS & CONFIGURATION ---
import sys
import re
import random
import warnings
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from google.colab import drive

# 1. Setup Environment & Drive
try:
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    print("Google Drive sudah ter-mount.")

# Setup Path Module
lib_path_modules = '/content/drive/Shareddrives/AFTERSALES EXTERNAL/SCRIPT/modules'
if lib_path_modules not in sys.path:
    sys.path.append(lib_path_modules)

try:
    import data_handler
except ImportError:
    print("Warning: Modul 'data_handler' tidak ditemukan di path.")

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1500)

# 2. Configuration Mappings

# Service Category Mapping (NEW)
SERVICE_TYPE_MAPPING = {
    'Interval Service': 'Regular',
    'Interval Service, Driver Coming (Trouble)': 'Regular',
    'Interval Service, Mechanic Visit (CS)': 'Regular',
    'Interval Service, Campaign (Big Issue)': 'Regular',
    'Mechanic Visit (CS), Campaign (Big Issue)': 'Regular',
    'Campaign (Big Issue), Mechanic Visit (CS)': 'Regular',
    'Campaign (Big Issue), Mechanic Visit (CS), Driver Coming (Trouble)': 'Regular',
    'Interval Service, Driver Coming (Trouble), Mechanic Visit (CS)': 'Regular',
    'Regular Service': 'Regular',
    'Dax In / Walk In (Service Reguler)': 'Regular',
    'Pause Unit': 'Urgent',
    'Driver Coming (Trouble)': 'Urgent',
    'Mechanic Visit (CS)': 'Storing',
    'Mechanic Visit (CS), Driver Coming Trouble (CS)': 'Storing',
    'Mechanic Visit (CS), Driver Coming (Trouble)': 'Storing',
    'ERA (Emergency Roadside Assistance) / Towing': 'Storing',
    'Repossesion': 'Repo Maintenance',
    'Reppo': 'Repo Maintenance',
    'Repo': 'Repo Maintenance',
    'repo maintenance': 'Repo Maintenance',
    'Walk In': 'Walk-In Maintenance',
    'Walk In/Offboarding': 'Walk-In Maintenance',
    'Broken Unit': 'Walk-In Maintenance',
    'Reduce NG': 'Walk-In Maintenance',
    'Campaign (Big Issue)': 'Storing',
    'Undangan DSS': 'Storing',
    'Unit Stock Grab': 'Walk-In Maintenance',
    'Reject QC': 'Urgent',
    'Driver Coming (Trouble), Mechanic Visit (CS), Campaign (Big Issue)' : 'Urgent',
    'Official Partner Service': 'Official Partner Service',
    'Warranty Claim': 'Regular',
    'Swap In (Tukar Unit)': 'Urgent',
    'Driver Coming (Trouble), Mechanic Visit (CS)': 'Urgent',
    'Resign / Offboarding': 'Walk-In Maintenance'
}

COLUMN_MAPPING = {
    'S1_FORM_SERVICE': {
        'Timestamp': 'created_at', 'Tanggal Service': 'completed_at', 'Lokasi Pool': 'service_location_name',
        'Nama Driver (1)': 'customer_name', 'Nama Driver': 'customer_name_backup',
        'ODO / KM': 'odometer',
        'Plate Number (1)': 'vehicle_license_plate', 'Plate Number': 'vehicle_license_plate_backup',
        'Plat Nomor': 'vehicle_license_plate_backup_2', 'Mechanic Name': 'completed_by',
        'Mechanic Action Category': 'service_type', 'Keluhan Driver': 'customer_problems',
        'Tindakan Dari Mekanik': 'action_description', 'Total Biaya Perbaikan': 'total_price'
    },
    'S2_SERVICE_GRAB': {
        'Timestamp': 'created_at', 'Tanggal': 'completed_at', 'Nama Mekanik': 'completed_by',
        'Plat Nomor': 'vehicle_license_plate', 'ODO / Kilometer': 'odometer',
        'Status Unit': 'service_type', 'Kendala Unit': 'customer_problems',
        'Lokasi Service': 'service_location_name'
    },
    'S3_FORM_RESPONSES': {
        'Timestamp': 'created_at', 'Lokasi Pool': 'service_location_name', 'Nama Driver': 'customer_name',
        'Tanggal Service': 'completed_at', 'Plate Number': 'vehicle_license_plate',
        'ODO / KM': 'odometer', 'Mechanic Action Category': 'service_type', 'Mechanic Name': 'completed_by',
        'Keluhan Driver': 'customer_problems', 'Tindakan dari Mekanik': 'action_description'
    },
    'S4_REQUEST_SPK': {
        'Tanggal Laporan': 'created_at', 'Nama Driver': 'customer_name',
        'Plat Nomor': 'vehicle_license_plate', 'Odo / Kilometer': 'odometer', 'Kendala Unit': 'customer_problems',
        'Bengkel Tujuan': 'service_location_name', 'Tanggal Service di Bengkel': 'completed_at'
    },
    'S5_AFTER_REPAIR': {
        'Tanggal': 'created_at', 'Plat Kendaraan / Vin': 'vehicle_license_plate',
        'Nama Mekanik': 'completed_by'
    }
}

PARTS_COLUMNS_MAPPING = {
    'S1_FORM_SERVICE': ['Nama Part yang diganti', 'Part lain yang diganti (1)', 'Part lain yang diganti (2)', 'Part lain yang diganti (3)', 'Part lain yang diganti (4)', 'Part lain yang diganti (5)'],
    'S2_SERVICE_GRAB': ['Consummable Part yang diganti', 'Sparepart lain yang diganti', '2. Part lain yang diganti', '3. Part lain yang diganti', '4. Part lain yang diganti', '5. Part lain yang diganti', 'Sparepart yang diganti'],
    'S3_FORM_RESPONSES': ['Fast Moving Part (H5)', 'Medium Moving Part (H5)', 'Slow Moving Part (H5)', 'Fast Moving Part (H3)', 'Medium Moving Part (H3)', 'Slow Moving Part (H3)', 'Fast Moving Part (H1)', 'Medium Moving Part (H1)', 'Slow Moving Part (H1)'],
    'S4_REQUEST_SPK': ['Nama Sparepart', 'Item Pengerjaan'],
    'S5_AFTER_REPAIR': ['Sparepart yang diganti - H3 ONLY', 'Sparepart yang diganti - H5 ONLY', 'Sparepart yang diganti - H1 ONLY'],
    'S6_KEMBANGAN': ['SparePart Changes', 'SparePart Name'],
    'S7_DEPOK': ['Consumable Part yang Diganti H3', 'Consumable Part yang Diganti H5', 'Consumable Part yang Diganti ALL', 'Part lain yang di ganti (1)', 'Part lain yang di ganti (2)', 'Part lain yang di ganti (3)', 'Part lain yang di ganti (4)', 'Part lain yang di ganti (5)'],
    'S8_BEKASI': ['Sparepart yang diganti - H3 ONLY', 'Sparepart yang diganti - H5 ONLY', 'ALL SPAREPART', 'BAHAN BAKU']
}

Mounted at /content/drive


#**UTILITIES (STATIC METHODS)**

In [34]:
# --- CELL 2: UTILITIES (STATIC METHODS) ---
import pandas as pd
import numpy as np
import re
import random
from datetime import datetime, timedelta

class ServiceUtils:

    @staticmethod
    def format_plat_nomor(plat_nomor):
        if not isinstance(plat_nomor, str):
            return None

        # Bersihkan karakter selain huruf dan angka
        plat_nomor_cleaned = re.sub(r'[^A-Za-z0-9]', '', plat_nomor).upper()

        if not plat_nomor_cleaned:
            return None

        plat_nomor_cleaned = plat_nomor_cleaned[:8]

        # Regex User
        match = re.match(r'^([A-Z])(\d{1,4})([A-Z]{0,3})$', plat_nomor_cleaned)

        if match:
            huruf_depan, angka, huruf_belakang = match.groups()
            formatted_plat = f"{huruf_depan} {angka}"
            if huruf_belakang:
                formatted_plat += f" {huruf_belakang}"
            return formatted_plat

        return None

    @staticmethod
    def combine_columns_to_string(row, columns_list):
        parts = []
        for col in columns_list:
            if col in row.index:
                val = row[col]
                if pd.notna(val) and str(val).strip() != '':
                    parts.append(str(val).strip())
        return ', '.join(parts)

    @staticmethod
    def randomize_work_hours(dt_val):
        """
        Mengubah jam 00:00:00 atau jam aneh menjadi jam kerja random (07:00 - 23:00).
        """
        if pd.isna(dt_val): return dt_val

        # Pastikan input adalah datetime
        if not isinstance(dt_val, (datetime, pd.Timestamp)):
            return dt_val

        # Cek komponen jam
        try:
            t = dt_val.time()
            # Logic: Jika jam persis 00:00:00 ATAU diluar range wajar (0-6 pagi)
            # Kita set ke jam kerja 07:00 - 23:00
            is_midnight = (t.hour == 0 and t.minute == 0 and t.second == 0)
            is_too_early = (t.hour < 7)

            if is_midnight or is_too_early:
                random_hour = random.randint(7, 23)
                random_minute = random.randint(0, 59)
                random_second = random.randint(0, 59)
                return dt_val.replace(hour=random_hour, minute=random_minute, second=random_second)
        except:
            pass

        return dt_val

    @staticmethod
    def fill_timeline(row):
        """
        Logika pintar mengisi timeline (Time Travel Logic).
        """
        created = row.get('created_at', pd.NaT)
        updated = row.get('updated_at', pd.NaT)
        completed = row.get('completed_at', pd.NaT)
        prize = row.get('prize_finalized_at', pd.NaT)

        def is_valid(dt): return pd.notna(dt)

        # 1. Tentukan Anchor
        anchor = None
        anchor_type = None

        if is_valid(completed):
            anchor = completed; anchor_type = 'completed'
        elif is_valid(created):
            anchor = created; anchor_type = 'created'
        elif is_valid(updated):
            anchor = updated; anchor_type = 'updated'

        if not anchor: return row

        # Pastikan anchor punya jam kerja yang masuk akal (07-23)
        anchor = ServiceUtils.randomize_work_hours(anchor)

        # 2. Logika Pengisian Relatif
        dur_service = timedelta(minutes=random.randint(30, 90))
        dur_start = timedelta(minutes=random.randint(5, 20))
        dur_admin = timedelta(minutes=random.randint(5, 15))

        if anchor_type == 'completed':
            completed = anchor
            if not is_valid(updated): updated = completed - dur_service
            if not is_valid(created): created = updated - dur_start
            if not is_valid(prize): prize = completed + dur_admin

        elif anchor_type == 'created':
            created = anchor
            if not is_valid(updated): updated = created + dur_start
            if not is_valid(completed): completed = updated + dur_service
            if not is_valid(prize): prize = completed + dur_admin

        elif anchor_type == 'updated':
            updated = anchor
            if not is_valid(created): created = updated - dur_start
            if not is_valid(completed): completed = updated + dur_service
            if not is_valid(prize): prize = completed + dur_admin

        row['created_at'] = created
        row['updated_at'] = updated
        row['completed_at'] = completed
        row['prize_finalized_at'] = prize

        return row

#**MAIN LOGIC CLASS (PIPELINE)**

In [35]:
# --- CELL 3: MAIN LOGIC CLASS (PIPELINE - DATE FIX) ---
import pandas as pd
import numpy as np

class ServiceDataPipeline:
    def __init__(self):
        self.asset_list = None
        self.raw_dfs = []
        self.master_data = pd.DataFrame()
        print("Pipeline Initialized.")

    def load_assets(self, spreadsheet, worksheet):
        print(f"Loading assets from {spreadsheet}...")
        try:
            self.asset_list = data_handler.load_gspread_data(spreadsheet, worksheet)
            print("Assets loaded successfully.")
        except Exception as e:
            print(f"Warning: Failed to load assets. {e}")

    def _robust_date_parse(self, series, source_name="Unknown"):
        """
        Helper: Parsing tanggal yang aman untuk tipe campuran (String & Timestamp).
        """
        if series.isna().all(): return pd.to_datetime(series)

        # DEBUG: Print sample
        print(f"   [{source_name}] Raw sample: {series.dropna().head(3).tolist()}")

        # 1. Jika kolom sudah datetime murni, return langsung
        if pd.api.types.is_datetime64_any_dtype(series):
            return series

        # 2. Parsing dengan handling mixed types
        # to_datetime dengan errors='coerce' biasanya cukup pintar menangani Timestamp object + String
        res = pd.to_datetime(series, errors='coerce', dayfirst=True)

        nat_count = res.isna().sum()
        total_count = len(res)
        nat_ratio = nat_count / total_count if total_count > 0 else 0

        # 3. Fallback jika banyak gagal (>30%)
        if nat_ratio > 0.3:
            print(f"   ⚠️ High NaT ({nat_ratio:.1%}). Retry parsing...")

            # Coba konversi ke string dulu baru parse (untuk format aneh)
            series_str = series.astype(str).str.strip()

            # Coba DayFirst=False (US)
            res_us = pd.to_datetime(series_str, errors='coerce', dayfirst=False)
            if res_us.isna().sum() < nat_count:
                print(f"   -> Fixed using DayFirst=False")
                return res_us

            # Coba format eksplisit
            formats = ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%d/%m/%Y %H:%M:%S']
            for fmt in formats:
                try:
                    res_fmt = pd.to_datetime(series_str, format=fmt, errors='coerce')
                    if res_fmt.isna().sum() < nat_count:
                        print(f"   -> Fixed using format: {fmt}")
                        return res_fmt
                except: continue

        return res

    def _clean_odometer(self, series):
        clean_series = series.astype(str).str.replace(r'[^\d]', '', regex=True).replace('', '0')
        return pd.to_numeric(clean_series, errors='coerce').fillna(0).astype('int64')

    def _assign_standard_columns(self, df, source_key):
        if df.empty: return df
        df.columns = df.columns.str.strip()

        if source_key in COLUMN_MAPPING:
            mapping = COLUMN_MAPPING[source_key]
            rename_dict = {}
            for k, v in mapping.items():
                if k in df.columns and v not in df.columns:
                    rename_dict[k] = v
            df = df.rename(columns=rename_dict)

        # Hardcode S5
        if source_key == 'S5_AFTER_REPAIR':
            # Cari kolom tanggal (case insensitive)
            if 'created_at' not in df.columns:
                candidates = [c for c in df.columns if 'tanggal' in c.lower()]
                if candidates:
                    print(f"   -> S5 Rename: '{candidates[0]}' to 'created_at'")
                    df = df.rename(columns={candidates[0]: 'created_at'})

        df = df.loc[:, ~df.columns.duplicated()]

        if source_key in PARTS_COLUMNS_MAPPING:
            valid_parts = [c for c in PARTS_COLUMNS_MAPPING[source_key] if c in df.columns]
            if valid_parts:
                df['item_name'] = df.apply(lambda r: ServiceUtils.combine_columns_to_string(r, valid_parts), axis=1)

        if source_key == 'S4_REQUEST_SPK': df['service_type'] = "Official Partner Service"
        elif source_key == 'S5_AFTER_REPAIR':
            df['service_type'] = "repo maintenance"
            if 'service_location_name' not in df.columns: df['service_location_name'] = "Pondok Indah"

        if 'service_type' in df.columns:
            df['service_type'] = df['service_type'].map(SERVICE_TYPE_MAPPING).fillna(df['service_type'])

        # Dates
        if 'created_at' in df.columns:
            df['created_at'] = self._robust_date_parse(df['created_at'], source_key)
        else:
            cands = [c for c in df.columns if any(x in str(c).lower() for x in ['timestamp', 'tanggal', 'date', 'time'])]
            if cands:
                print(f"   -> Auto-detect date ({source_key}): {cands[0]}")
                df['created_at'] = self._robust_date_parse(df[cands[0]], source_key)
            else:
                print(f"   ❌ CRITICAL: No date column found for {source_key}. Columns: {list(df.columns)}")

        for col in ['updated_at', 'completed_at', 'prize_finalized_at']:
            if col in df.columns: df[col] = self._robust_date_parse(df[col], f"{source_key}-{col}")

        # Timeline Fill
        df = df.apply(ServiceUtils.fill_timeline, axis=1)

        if 'order_status' not in df.columns: df['order_status'] = 'COMPLETED'
        if 'odometer' in df.columns: df['odometer'] = self._clean_odometer(df['odometer'])
        else: df['odometer'] = 0

        if 'vehicle_license_plate' not in df.columns: df['vehicle_license_plate'] = np.nan
        for backup in ['vehicle_license_plate_backup', 'vehicle_license_plate_backup_2']:
            if backup in df.columns:
                df['vehicle_license_plate'] = df['vehicle_license_plate'].fillna(df[backup])

        df['vehicle_license_plate'] = df['vehicle_license_plate'].apply(ServiceUtils.format_plat_nomor)

        return df

    def _process_cabang_logic(self, df, location_name, source_key):
        if df.empty: return df
        df.columns = df.columns.str.strip()

        if source_key in PARTS_COLUMNS_MAPPING:
            valid_cols = [c for c in PARTS_COLUMNS_MAPPING[source_key] if c in df.columns]
            if valid_cols:
                df['item_name'] = df.apply(lambda r: ServiceUtils.combine_columns_to_string(r, valid_cols), axis=1)

        # Smart Rename
        time_priority = ['Timestamp', 'Date', 'Tanggal']
        renamed = False
        for col in time_priority:
            if col in df.columns:
                df = df.rename(columns={col: 'created_at'})
                print(f"   -> Cabang {location_name}: '{col}' mapped to 'created_at'")
                renamed = True
                break

        if not renamed:
             cands = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
             if cands:
                 print(f"   -> Cabang {location_name}: Auto-found '{cands[0]}' as created_at")
                 df = df.rename(columns={cands[0]: 'created_at'})

        col_map = {
            'SparePart Name': 'item_name', 'Plate Number': 'vehicle_license_plate',
            'Plat Nomor': 'vehicle_license_plate', 'Plat Number': 'vehicle_license_plate',
            'Plat Kendaraan / VIN': 'vehicle_license_plate',
            'Date Bike Repair': 'completed_at',
            'Problem': 'customer_problems', 'Repair Action': 'action_description',
            'Mechanic Name': 'completed_by', 'Checker / Repair': 'completed_by', 'Nama Mekani': 'completed_by'
        }
        for k in ['Timestamp', 'Date', 'Tanggal']:
            if k in col_map: del col_map[k]

        df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})
        df = df.loc[:, ~df.columns.duplicated()]

        if 'vehicle_license_plate' in df.columns:
            df['vehicle_license_plate'] = df['vehicle_license_plate'].astype(str).apply(ServiceUtils.format_plat_nomor)

        df['service_location_name'] = location_name
        df['order_status'] = 'COMPLETED'

        if source_key == 'S6_KEMBANGAN': df['service_type'] = "repo maintenance"
        else: df['service_type'] = "Walk-In Maintenance"

        if 'service_type' in df.columns:
            df['service_type'] = df['service_type'].map(SERVICE_TYPE_MAPPING).fillna(df['service_type'])

        # Dates
        if 'created_at' in df.columns:
             df['created_at'] = self._robust_date_parse(df['created_at'], source_key)
        else:
             print(f"   ❌ Cabang {location_name}: No 'created_at' column found!")

        if 'completed_at' in df.columns: df['completed_at'] = self._robust_date_parse(df['completed_at'], f"{source_key}-completed")

        if 'odometer' not in df.columns: df['odometer'] = 0
        else: df['odometer'] = self._clean_odometer(df['odometer'])

        df = df.apply(ServiceUtils.fill_timeline, axis=1)
        return df

    def _append_to_raw(self, df, source_key):
        df = df.loc[:, ~df.columns.duplicated()]
        final_cols = ['created_at', 'updated_at', 'completed_at', 'prize_finalized_at', 'vehicle_license_plate', 'odometer', 'service_location_name', 'bike_type', 'item_name', 'customer_name', 'service_type', 'order_status', 'customer_problems', 'action_description', 'completed_by', 'total_price', 'data_source', 'vechicle_vin', 'vechicle_engine', 'color', 'customer_type']
        for c in final_cols:
            if c not in df.columns: df[c] = None
        df['data_source'] = source_key
        self.raw_dfs.append(df[final_cols])

    def ingest_generic(self, source_key, spreadsheet_name, sheet_name, is_csv=False, filter_func=None):
        print(f"Processing {source_key}...")
        try:
            if is_csv: df = data_handler.load_csv(spreadsheet_name, sheet_name)
            else: df = data_handler.load_gspread_data(spreadsheet_name, sheet_name)
        except Exception as e:
            print(f"Error loading {source_key}: {e}")
            return

        if df is None or df.empty: return
        if filter_func: df = filter_func(df)
        df = self._assign_standard_columns(df, source_key)
        self._append_to_raw(df, source_key)

    def ingest_cabang(self, source_key, spreadsheet_name, sheet_name, location_name):
        print(f"Processing Cabang {location_name} ({source_key})...")
        try:
            df = data_handler.load_gspread_data(spreadsheet_name, sheet_name)
        except Exception as e:
            print(f"Error: {e}")
            return

        if df is None or df.empty: return
        df = self._process_cabang_logic(df, location_name, source_key)
        self._append_to_raw(df, source_key)

    def merge_and_finalize(self):
        print("Merging all data sources...")
        if not self.raw_dfs: return
        self.master_data = pd.concat(self.raw_dfs, ignore_index=True)
        if 'odometer' in self.master_data.columns:
             self.master_data['odometer'] = self._clean_odometer(self.master_data['odometer'])
        for col in ['created_at', 'updated_at', 'completed_at', 'prize_finalized_at']:
             self.master_data[col] = pd.to_datetime(self.master_data[col], errors='coerce')
        self.master_data.sort_values(by='created_at', inplace=True)
        print(f"Total rows merged: {len(self.master_data)}")

##**ENRICHMENT & BUSINESS LOGIC**

In [36]:
# --- CELL 4: ENRICHMENT & BUSINESS LOGIC (SAFE ODO & BAD DATA LOG) ---
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher
from datetime import datetime

class ServiceDataEnricher:
    def __init__(self, master_data, asset_list_df):
        self.df = master_data.copy()
        self.assets = asset_list_df.copy() if asset_list_df is not None else pd.DataFrame()
        self.bad_data = pd.DataFrame()

        if 'Plat Nomor' in self.assets.columns:
            self.assets['Plat_Clean'] = self.assets['Plat Nomor'].astype(str).apply(ServiceUtils.format_plat_nomor).fillna('').str.strip().str.upper()

            def get_col(candidates):
                for c in candidates:
                    if c in self.assets.columns: return c
                return None

            col_model = get_col(['Model', 'model', 'Bike Model'])
            col_sewa = get_col(['Tempat Sewa Unit', 'tempat sewa unit', 'Sewa', 'Customer Type'])
            col_color = get_col(['Color', 'color', 'Warna', 'Colour'])

            valid_assets = self.assets[self.assets['Plat_Clean'] != '']
            self.vin_map = valid_assets.set_index('Plat_Clean')['VIN'].to_dict()
            self.engine_map = valid_assets.set_index('Plat_Clean')['Engine No'].to_dict()
            self.model_map = valid_assets.set_index('Plat_Clean')[col_model].to_dict() if col_model else {}
            self.sewa_map = valid_assets.set_index('Plat_Clean')[col_sewa].to_dict() if col_sewa else {}
            self.color_map = valid_assets.set_index('Plat_Clean')[col_color].to_dict() if col_color else {}
        else:
            self.vin_map = {}; self.engine_map = {}; self.model_map = {}; self.sewa_map = {}; self.color_map = {}

    def _log_bad_data(self, bad_rows, reason):
        if not bad_rows.empty:
            bad_rows = bad_rows.copy()
            bad_rows['reject_reason'] = reason
            self.bad_data = pd.concat([self.bad_data, bad_rows], ignore_index=True)

    def _fuzzy_match_plate(self, bad_plate, threshold=0.85):
        if pd.isna(bad_plate) or len(str(bad_plate)) < 4: return None
        best_match, best_score = None, 0
        target_len = len(bad_plate)

        candidates = [p for p in self.vin_map.keys() if isinstance(p, str) and abs(len(p) - target_len) <= 1]

        for asset_plate in candidates:
            score = SequenceMatcher(None, bad_plate, asset_plate).ratio()
            if score > best_score:
                best_score, best_match = score, asset_plate
        return best_match if best_score >= threshold else None

    def clean_critical_data(self):
        """Req 1: Log & Remove missing created_at"""
        print("   - Cleaning data without created_at...")
        mask_missing = self.df['created_at'].isna()
        self._log_bad_data(self.df[mask_missing], "Missing Timestamp")

        len_before = len(self.df)
        self.df = self.df[~mask_missing].reset_index(drop=True)
        print(f"     Dropped {len_before - len(self.df)} rows.")
        return self

    def repair_identities_and_clean(self):
        """Req 2: Repair Identities & Fuzzy Match"""
        print("🚀 Repairing Identities & Cleaning Bad Data...")

        for c in ['vechicle_vin', 'vechicle_engine', 'color', 'customer_type', 'bike_type']:
            if c not in self.df.columns: self.df[c] = np.nan

        self.df['plat_clean'] = self.df['vehicle_license_plate'].astype(str).apply(ServiceUtils.format_plat_nomor).fillna('').str.strip().str.upper()

        # 1. Exact Match
        self.df['vechicle_vin'] = self.df['vechicle_vin'].fillna(self.df['plat_clean'].map(self.vin_map))
        self.df['vechicle_engine'] = self.df['vechicle_engine'].fillna(self.df['plat_clean'].map(self.engine_map))
        self.df['color'] = self.df['color'].fillna(self.df['plat_clean'].map(self.color_map))

        # 2. Fuzzy Match
        mask_still_missing = self.df['vechicle_vin'].isna()
        unique_missing = self.df.loc[mask_still_missing, 'plat_clean'].unique()

        corrected_map = {}
        for bad_plate in unique_missing:
            if not bad_plate or len(bad_plate) < 4: continue
            match = self._fuzzy_match_plate(bad_plate)
            if match: corrected_map[bad_plate] = match

        if corrected_map:
            print(f"   - Correcting {len(corrected_map)} typo plates...")
            for bad, good in corrected_map.items():
                mask_fix = self.df['plat_clean'] == bad
                self.df.loc[mask_fix, 'vehicle_license_plate'] = good
                self.df.loc[mask_fix, 'plat_clean'] = good
                self.df.loc[mask_fix, 'vechicle_vin'] = self.vin_map.get(good)
                self.df.loc[mask_fix, 'vechicle_engine'] = self.engine_map.get(good)
                self.df.loc[mask_fix, 'color'] = self.color_map.get(good)

        # 3. Log Bad Data
        mask_invalid = self.df['vechicle_vin'].isna()
        self._log_bad_data(self.df[mask_invalid], "Invalid/Unknown License Plate")

        n_dropped = mask_invalid.sum()
        if n_dropped > 0:
            self.df = self.df[~mask_invalid].reset_index(drop=True)
            print(f"     Dropped {n_dropped} rows (Unrecognized License Plate).")

        self.df.drop(columns=['plat_clean'], inplace=True)
        print(f"✅ Identity Repair Finished. Active Rows: {len(self.df)}")
        return self

    def enrich_asset_details(self):
        """Req 3: Fill Details"""
        print("🚀 Enriching Asset Details...")
        self.df['plat_clean'] = self.df['vehicle_license_plate'].astype(str).str.strip().str.upper()

        self.df['bike_type'] = self.df['bike_type'].fillna(self.df['plat_clean'].map(self.model_map))
        self.df['customer_type'] = self.df['customer_type'].fillna(self.df['plat_clean'].map(self.sewa_map))
        self.df['color'] = self.df['color'].fillna(self.df['plat_clean'].map(self.color_map))

        self.df.drop(columns=['plat_clean'], inplace=True)
        return self

    def fill_customer_names(self):
        """Req 5 & 6: Backfill & Fallback"""
        print("🚀 Backfilling Customer Names...")
        self.df.sort_values(['vehicle_license_plate', 'created_at'], inplace=True)

        self.df['customer_name'] = self.df.groupby('vehicle_license_plate')['customer_name'].ffill().bfill()

        def fallback_name(row):
            if pd.notna(row['customer_name']): return row['customer_name']
            ctype = str(row['customer_type']).upper()
            if 'GEL' in ctype: return "Driver GEL"
            if 'DAX' in ctype or 'GRAB' in ctype: return "Driver Grab"
            return "Sahabat Electrum"

        mask_empty = self.df['customer_name'].isna()
        if mask_empty.any():
            self.df.loc[mask_empty, 'customer_name'] = self.df.loc[mask_empty].apply(fallback_name, axis=1)

        return self

    def process_odometer(self):
        """
        Req 4:
        1. Estimasi ODO 0 (Based on history).
        2. Keep Original Value (No rounding/truncating).
        """
        print("🚀 Processing Odometer (Keep Original Value)...")

        # Pastikan Numeric Murni & Handle NaN
        if 'odometer' not in self.df.columns: self.df['odometer'] = 0
        self.df['odometer'] = pd.to_numeric(self.df['odometer'], errors='coerce').fillna(0)

        # 1. Estimasi ODO 0
        self.df.sort_values(by=['vehicle_license_plate', 'created_at'], inplace=True)
        temp_odo = self.df['odometer'].replace(0, np.nan)
        last_val = temp_odo.ffill()

        last_date = self.df['created_at'].where(self.df['odometer'] > 0).ffill()
        last_plate = self.df['vehicle_license_plate'].where(self.df['odometer'] > 0).ffill()

        diff_days = (self.df['created_at'] - last_date).dt.days
        mask_est = (self.df['odometer'] == 0) & (last_val.notna()) & (last_plate == self.df['vehicle_license_plate']) & (diff_days >= 0)

        self.df.loc[mask_est, 'odometer'] = last_val + (diff_days * 100)

        # 2. Force Integer (Hanya casting, tanpa ubah nilai)
        self.df['odometer'] = self.df['odometer'].astype('int64')

        print("✅ Odometer Processed.")
        return self

    def standardize_mechanics(self, employee_df):
        print("🚀 Standardizing Mechanics...")
        if employee_df is None or employee_df.empty: return self
        patterns = []
        for _, row in employee_df.iterrows():
            if pd.notna(row['Pola Regex']):
                try: patterns.append((re.compile(r'\b' + str(row['Pola Regex']) + r'\b', re.IGNORECASE), row['Full Name']))
                except: continue

        def _match(name, stype):
            name = str(name).strip()
            if stype == 'Official Partner Service': return "Mechanic Workshop Partner"
            if len(name) < 3 or name.lower() in ['nan', 'none', '']: return "Daily Worker"
            for p, full in patterns:
                if p.search(name): return full
            return "Daily Worker"

        self.df['completed_by'] = self.df.apply(lambda x: _match(x['completed_by'], x['service_type']), axis=1)
        return self

    def generate_snowflake_ids(self, worker_id=1, datacenter_id=1):
        print("🚀 Generating IDs...")
        EPOCH = 1704067200000; SEQUENCE = 0
        def _get_id(dt):
            nonlocal SEQUENCE
            if pd.isna(dt): dt = datetime.now()
            ts = int(dt.timestamp() * 1000) - EPOCH
            if ts < 0: ts = 0
            SEQUENCE = (SEQUENCE + 1) % 4095
            sid = (ts << 22) | (datacenter_id << 17) | (worker_id << 12) | SEQUENCE
            return f"WO-{sid}"
        self.df['order_id'] = self.df['created_at'].apply(_get_id)
        return self

    def get_results(self):
        return self.df, self.bad_data

#**FINAL EXECUTION (ORCHESTRATOR)**

In [37]:
# --- CELL 5: FINAL EXECUTION ---
import data_handler
import pandas as pd

# 1. INGESTION
print("=== TAHAP 1: INGESTION ===")
pipeline = ServiceDataPipeline()
pipeline.load_assets("List All Bike SCM", "ALL BIKE NEW")
asset_df = pipeline.asset_list

# Ingest Data Sources
pipeline.ingest_generic("S1_FORM_SERVICE", "FORM SERVICE ELECTRUM 2025 (26June) (Responses)", "Service History (2024-Jun 2025)")
pipeline.ingest_generic("S2_SERVICE_GRAB", "Form Service Unit Grab - Electrum (Responses)", "Form responses 1")
pipeline.ingest_generic("S3_FORM_RESPONSES", "FORM SERVICE ELECTRUM 2025 (26June) (Responses)", "Form Responses 1")

# Filter S4
def s4_filter(df):
    return df[
        df['Status'].astype(str).str.contains('Completed', case=False, na=False) &
        df['Bengkel Tujuan'].notna()
    ]
pipeline.ingest_generic("S4_REQUEST_SPK", "LIST REQUEST SPK", "DATA 2025", filter_func=s4_filter)

# DEBUG CHECK S5
pipeline.ingest_generic("S5_AFTER_REPAIR", "BREAKDOWN NG", "List After Repair")

# Ingest Cabang (Check S8 Bekasi)
pipeline.ingest_cabang("S6_KEMBANGAN", "BREAKDOWN NG", "Repair Kmb", location_name="Kembangan")
pipeline.ingest_cabang("S7_DEPOK", "Tracker Sparepart WH Depok", "Daily Repair 1", location_name="Depok")
pipeline.ingest_cabang("S8_BEKASI", "form Repair Bike - Bekasi (Responses)", "Form responses 1", location_name="Bekasi")

pipeline.merge_and_finalize()
merged_df = pipeline.master_data

# DEBUG: Print data kosong per source
print("\n🔍 Analisis Missing Date (After Ingestion):")
missing_time = merged_df[merged_df['created_at'].isna()]
if not missing_time.empty:
    print(missing_time['data_source'].value_counts())
else:
    print("✅ All created_at are valid!")

# 2. ENRICHMENT
print("\n=== TAHAP 2: ENRICHMENT & CLEANING ===")
try: employee_df = data_handler.load_gspread_data("DATA KARYAWAN", "mekanik list")
except: employee_df = None

enricher = ServiceDataEnricher(merged_df, asset_df)

final_df, df_bad_data = (
    enricher
    .clean_critical_data()
    .repair_identities_and_clean()
    .enrich_asset_details()
    .fill_customer_names()
    .process_odometer()
    .standardize_mechanics(employee_df)
    .generate_snowflake_ids()
    .get_results()
)

# 3. OUTPUT
print("\n" + "="*40)
print(f"Final Data Rows: {len(final_df)}")
print("-" * 40)
display(final_df[['created_at', 'data_source', 'vehicle_license_plate']].head(5))

=== TAHAP 1: INGESTION ===
Pipeline Initialized.
Loading assets from List All Bike SCM...
Assets loaded successfully.
Processing S1_FORM_SERVICE...
   [S1_FORM_SERVICE] Sample raw dates: [Timestamp('2024-08-21 10:50:32'), Timestamp('2024-08-21 10:54:59'), Timestamp('2024-08-21 13:25:09')]
   [S1_FORM_SERVICE-completed_at] Sample raw dates: [Timestamp('2024-08-21 00:00:00'), Timestamp('2024-08-21 00:00:00'), Timestamp('2024-08-21 00:00:00')]
Processing S2_SERVICE_GRAB...
   [S2_SERVICE_GRAB] Sample raw dates: [Timestamp('2025-01-25 15:38:19'), Timestamp('2024-12-18 15:02:45'), Timestamp('2025-01-03 15:09:11')]
   [S2_SERVICE_GRAB-completed_at] Sample raw dates: [Timestamp('2024-11-18 00:00:00'), Timestamp('2024-11-20 00:00:00'), Timestamp('2024-11-20 00:00:00')]
Processing S3_FORM_RESPONSES...
   [S3_FORM_RESPONSES] Sample raw dates: [Timestamp('2025-01-07 10:51:36'), Timestamp('2025-01-07 11:21:56'), Timestamp('2025-01-07 11:25:49')]
   ⚠️ High NaT rate (57.1%) in S3_FORM_RESPONSES. Tr

,created_at,data_source,vehicle_license_plate
22286,2025-06-30 00:00:00,S4_REQUEST_SPK,B 3077 WWC
39737,2025-11-21 11:47:51,S3_FORM_RESPONSES,B 3077 WWC
11800,2025-03-14 17:29:42,S2_SERVICE_GRAB,B 3403 PWG
20089,2025-06-10 15:30:34,S2_SERVICE_GRAB,B 3403 PWG
18551,2025-05-27 15:48:51,S1_FORM_SERVICE,B 3404 PWG


In [38]:
# Export Data
data_handler.clean_and_upload_to_google_sheet(
    final_df,
    "1DiCivMEoFNDQxlaGIb66VdsVN9jsE-qkpQF2VJ28Xok",
    "work_orders"
)

Worksheet 'work_orders' telah dibersihkan.
Data berhasil diunggah (overwrite) ke worksheet 'work_orders'.


In [39]:
# Export Data
data_handler.clean_and_upload_to_google_sheet(
    df_bad_data,
    "1DiCivMEoFNDQxlaGIb66VdsVN9jsE-qkpQF2VJ28Xok",
    "bad_data"
)

Worksheet 'bad_data' telah dibersihkan.
Data berhasil diunggah (overwrite) ke worksheet 'bad_data'.
